# AI Cartoon Studio — local LLM server
This notebook runs a self-hosted OpenAI-compatible vLLM endpoint. Colab is only the inference worker; the application database and dashboard stay elsewhere.


In [ ]:
!nvidia-smi
!pip -q install 'vllm>=0.6' requests


In [ ]:
import os, secrets
MODEL_ID = os.getenv('MODEL_ID', 'Qwen/Qwen2.5-7B-Instruct-AWQ')
API_KEY = os.getenv('LLM_API_KEY', secrets.token_urlsafe(32))
PORT = 8001
print('Model:', MODEL_ID)
print('Temporary API key:', API_KEY)


In [ ]:
import subprocess, sys, time
command = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
           '--model', MODEL_ID, '--host', '0.0.0.0', '--port', str(PORT),
           '--api-key', API_KEY, '--trust-remote-code']
server = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(20)
print('Server process:', server.pid)


In [ ]:
import requests, time
url = f'http://127.0.0.1:{PORT}/v1/models'
for _ in range(60):
    try:
        response = requests.get(url, headers={'Authorization': f'Bearer {API_KEY}'}, timeout=5)
        if response.ok:
            print(response.json())
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise RuntimeError('vLLM did not become ready')


## Optional temporary tunnel
Use this only for private development. The generated URL changes whenever the session restarts.


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess, re
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in tunnel.stdout:
    print(line, end='')
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
print('LLM_BASE_URL=', f'{public_url}/v1' if public_url else 'Tunnel URL not found')
print('LLM_API_KEY=', API_KEY)
print('LLM_MODEL=', MODEL_ID)
